# 03 · Robustness under visual perturbations (Step 2a/2b)

<a href="https://colab.research.google.com/github/danielamrh/act-visual-robustness/blob/main/notebooks/03_robustness.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

Evaluates a policy under the perturbation suite in [`src/avr/perturb`](../src/avr/perturb):
11 factors (lighting, table color, background, distractors, camera pose, pixel noise, blur, JPEG, cube color)
x 4 severity levels. The perturbations are **purely visual**: unit tests check that robot and cube
physics are bit-identical to the unperturbed scene, only the camera image changes.
Each episode's perturbation is seeded by the episode seed, so every policy sees exactly the same
perturbed episodes (paired comparison).

| Section | What |
|---|---|
| 2 | Preview grid of all factors and levels |
| 3 | Validate our rollout runner against `lerobot-eval` (same seeds, same episodes) |
| 4 | Full suite on a checkpoint (resumable, runs in the background) |
| 5 | Results table + robustness curves |

**Colab Free:** only one GPU runtime at a time - run this after training in notebook 02 has finished
(or stopped at a checkpoint; it resumes later).

## 0 · Environment check

In [ ]:
import sys, subprocess
print(sys.version)
assert sys.version_info >= (3, 12), "LeRobot 0.6.x needs Python >= 3.12"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "No GPU - switch the runtime to T4!")

## 1 · Drive + repo + install
Code comes from GitHub (always fresh via `git pull`), outputs go to Google Drive.

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")

REPO = "act-visual-robustness"
REPO_DIR = f"/content/{REPO}"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/danielamrh/{REPO}.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
COMMIT = !git rev-parse --short HEAD
COMMIT = COMMIT[0]
print("commit:", COMMIT)

# labmaze (pulled in by dm-control) has no wheel for Python >= 3.13 and needs Bazel
# to build. gym-aloha never uses it, so install an empty placeholder first.
if sys.version_info >= (3, 13):
    !pip install -q ./tools/labmaze_stub

# installs lerobot[aloha] pinned in pyproject.toml (takes a few minutes)
!pip install -q -e ".[sim]"
sys.path.insert(0, f"{REPO_DIR}/src")  # editable install is only picked up after a restart
# re-running this cell in a live session must pick up the freshly pulled code:
# drop already imported avr modules so the next import loads the new version
for name in [m for m in sys.modules if m == "avr" or m.startswith("avr.")]:
    del sys.modules[name]

# GPU rendering: without NVIDIA's EGL registration MuJoCo silently renders on the CPU
from avr.colab import ensure_nvidia_egl, gl_renderer
ensure_nvidia_egl()
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
renderer = gl_renderer()
print("OpenGL renderer:", renderer)
if "llvmpipe" in renderer.lower() or renderer.startswith("failed"):
    print("⚠ MuJoCo is NOT rendering on the GPU - rollouts will be extremely slow")

DRIVE_ROOT = "/content/drive/MyDrive/act_robustness"
os.makedirs(DRIVE_ROOT, exist_ok=True)

### Quick check: can we create and render the ALOHA env?

In [ ]:
import gymnasium as gym
import gym_aloha  # registers gym_aloha/* envs
import matplotlib.pyplot as plt

import time
env = gym.make("gym_aloha/AlohaTransferCube-v0")
obs, _ = env.reset(seed=0)
img = obs["top"] if isinstance(obs, dict) and "top" in obs else env.render()

# speed check: env step incl. observation rendering, zero actions
t0 = time.perf_counter()
for _ in range(50):
    env.step(env.action_space.sample() * 0)
dt = (time.perf_counter() - t0) / 50
env.close()
print(f"{dt*1000:.0f} ms per env step -> one 400-step episode ≈ {400*dt:.0f} s (+ video rendering)")
plt.imshow(img); plt.axis("off"); plt.title("AlohaTransferCube-v0, seed 0");

## 2 · Preview: one row per factor, one column per severity level (seed 0)

In [ ]:
from IPython.display import display
from avr.perturb.preview import preview_grid

ROB_BASE = f"{DRIVE_ROOT}/robustness"
os.makedirs(ROB_BASE, exist_ok=True)
grid = preview_grid(seed=0, scale=0.3)
grid.save(f"{ROB_BASE}/preview_seed0.png")
display(grid)

## 3 · Validate the runner against `lerobot-eval`
Our runner steps `PerturbedAlohaEnv` itself (needed for the perturbations). With no perturbation and the same
seeds (1000-1099) it should reproduce the episodes of notebook 01's first chunk. Small differences are possible
(batch-of-5 vs. single-episode GPU numerics), large ones would mean a bug.

In [ ]:
import json
from avr.background import start_background, watch
from avr.eval.chunked import chunk_info_paths
from avr.eval.stats import merge_eval_infos

POLICY_DIR = f"{DRIVE_ROOT}/checkpoints/act_aloha_sim_transfer_cube_human_migrated"
ROB_ROOT = f"{ROB_BASE}/pretrained_resnet18"

def run_suite(policy, out, factors, levels=(1, 2, 3, 4), episodes=50, seed=1000, tag="suite"):
    cmd = [sys.executable, "-m", "avr.eval.robustness", "--policy", policy, "--out", out,
           "--factors", *factors, "--levels", *map(str, levels),
           "--episodes", str(episodes), "--seed", str(seed)]
    log = f"/content/logs/{os.path.basename(out)}_{tag}.log"
    watch(start_background(cmd, log), log, interval=60)

run_suite(POLICY_DIR, ROB_ROOT, ["clean"], episodes=100, tag="clean")

ours = json.load(open(f"{ROB_ROOT}/clean/level0.json"))["episodes"]
ref = merge_eval_infos(chunk_info_paths(f"{DRIVE_ROOT}/eval/pretrained_transfer_cube/n500_seed1000")[:1])["per_episode"]
agree = sum(a["success"] == b["success"] for a, b in zip(ours, ref))
print(f"runner: {sum(e['success'] for e in ours)}/{len(ours)}   lerobot-eval: {sum(e['success'] for e in ref)}/{len(ref)}")
print(f"same outcome in {agree}/{len(ours)} paired episodes")

## 4 · Full suite (11 factors x 4 levels x 50 episodes)
Seeds 1000-1049 for every cell. Finished cells are stored on Drive as `<factor>/level<k>.json` and skipped
on re-run, so the suite can be spread over several sessions. Two videos per cell are saved to `videos/`.
The status line shows the running success rate of the current cell.

In [ ]:
from avr.perturb import FACTORS
run_suite(POLICY_DIR, ROB_ROOT, sorted(FACTORS), episodes=50)

## 5 · Results

In [ ]:
import pandas as pd
from avr.analysis.robustness import load_cells, plot_robustness, summary_rows

cells = load_cells(ROB_ROOT)
clean_rate = cells[("clean", 0)]["episodes"]
clean_rate = sum(e["success"] for e in clean_rate) / len(clean_rate)

df = pd.DataFrame(summary_rows(cells))
df.to_csv(f"{ROB_ROOT}/summary.csv", index=False)
pct = {c: "{:.0%}" for c in ["success", "ci_lo", "ci_hi", "fail_no_touch", "fail_grasp", "fail_transport"]}
display(df.style.format(pct).hide(axis="index"))

label = "pretrained ACT (ResNet18)"
fig = plot_robustness({label: cells}, {label: clean_rate}, path=f"{ROB_ROOT}/robustness_curves.png")

## 6 · Failure videos

In [ ]:
import glob
from IPython.display import Video
fails = sorted(glob.glob(f"{ROB_ROOT}/videos/*_fail.mp4"))
print(len(fails), "failure videos, e.g.:", *[os.path.basename(f) for f in fails[:10]], sep="\n  ")
Video(fails[0], embed=True, width=480) if fails else None